<a href="https://colab.research.google.com/github/mk654/SML_PG60/blob/main/COMP90051_ProjectGroup60_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# **COMP90051 Group Project → Code**

| Project Group 60 |         |
|------------------|---------|
| Lachlan Fox      | 649622  |
| Songhao Guo      | 1542657 |
| Amelia King      | 1175861 |


github → https://github.com/mk654/SML_PG60


In [ ]:
# RUN FIRST
from scipy.io import loadmat
import pandas as pd
from pathlib import Path
import numpy as np
from scipy import sparse


# Amelia -> load data from github repo
!git clone https://github.com/mk654/SML_PG60
repo_dir = Path("/content/SML_PG60")

flu_mat = repo_dir / "data" / "matraw" / "influenza_outbreak_dataset.mat" # access influenza dataset from gitrepo
fludata = loadmat(flu_mat)



# url = "https://raw.githubusercontent.com/mk654/SML_PG60/main/influenza_outbreak_dataset.mat" # old access -> directory = main
#url = "https://raw.githubusercontent.com/mk654/SML_PG60/main/data/influenza_outbreak_dataset.mat" # old access -> directory = main/data


### 0.a) inspecting data

*Amelia*


#### *Results*
| name     | outer dtype | outer shape | inner type | inner shape |
|----------|-------------|-------------|------------|-------------|
| X train  | object      | (1, 48)     | csc_matrix | (1095, 545) |
| X test   | object      | (1, 48)     | csc_matrix | (485, 545)  |
| y train  | object      | (1, 48)     | ndarray    | (1095, 1)   |
| y test   | object      | (1, 48)     | ndarray    | (485, 1)    |
| locs     | object      | (1, 48)     | ndarray    | (1,)        |
| keywords | object      | (1, 525)    | ndarray    | (1,)        |

in sum:
- 48 training feature matrices, one per location
- 48 testing feature matrices, one per location
- 48 training label vectors
- 48 testing label vectors
- 48 location names/IDs
- 545 keyword feature names



*Code*


```
rows = []
for name in ["flu_X_tr", "flu_X_te", "flu_Y_tr", "flu_Y_te", "flu_locs", "flu_keywords"]:
    value = fludata[name]
    try:
        first = value[0, 0]
        first_type = type(first).__name__
        first_shape = getattr(first, 'shape', None)
    except:
        first_shape = None
        first_type = None
    rows.append({
        "name": name,
        "outer_dtype": value.dtype,
        "outer_shape": value.shape,
        "inner_type": first_type,
        "inner_shape": first_shape
    })
df_summary = pd.DataFrame(rows)
print(df_summary)
```

In [ ]:
# delete this cell before submission!!

rows = []
for name in ["flu_X_tr", "flu_X_te", "flu_Y_tr", "flu_Y_te", "flu_locs", "flu_keywords"]:
    value = fludata[name]
    try:
        first = value[0, 0]
        first_type = type(first).__name__
        first_shape = getattr(first, 'shape', None)
    except:
        first_shape = None
        first_type = None
    rows.append({
        "name": name,
        "outer_dtype": value.dtype,
        "outer_shape": value.shape,
        "inner_type": first_type,
        "inner_shape": first_shape
    })
df_summary = pd.DataFrame(rows)
print(df_summary)

### 0.b) convert .mat files to .csv
Converts influenza_outbreak_dataset.mat to csv

github directory = ```SML_PG60/data/processed/flu_csv```

----
*Note that these csv files are only processed in the sense that they have been converted from .mat to .csv; no further processing has yet taken place.*

### Amelia's conversion
- conversion will produce 48 separate folders aligning with the 48 folds. Each folder will contain:
  1. X_train.csv
  2. X_test.csv
  3. y_train.csv
  4. y_test.csv
  * *note that influenza_outbreak_dataset.mat contains 48 folds (test/train splits). Each fold contains its own X & y train and X & y test. Hence, the data is split and converted as cleanly as possible to avoid errors that may come with combining folds*
- Additionally, the conversion will also produce:
  1. keywords.csv
  2. locs.csv

---


The converted files can be manually downloaded from this notebook by copy/pasting & running the following code:

*this isn't currently working :(*
```
from google.colab import files
!zip -r ak_flucsv.zip /content/SML_PG60/data/processed/flu_csv/"ak_flucsv"
files.download("ak_flucsv.zip")
```

In [ ]:
# Amelia -> convert influenza_outbreak_dataset.mat to .csv file
flu_out = repo_dir / "data" / "processed" / "flu_csv" / "ak_flucsv"
flu_out.mkdir(parents=True, exist_ok=True)

X_tr = fludata["flu_X_tr"]
X_te = fludata["flu_X_te"]
y_tr = fludata["flu_Y_tr"]
y_te = fludata["flu_Y_te"]

n_folds = X_tr.shape[1]

for i in range(n_folds):
    fold_dir = flu_out / f"fold_{i:02d}"
    fold_dir.mkdir(exist_ok=True)

    Xtr = X_tr[0, i].toarray() # some data stored as sparse matrix, convert to dense
    Xte = X_te[0, i].toarray()
    ytr = y_tr[0, i].ravel()
    yte = y_te[0, i].ravel()

    pd.DataFrame(Xtr).to_csv(fold_dir / "X_train.csv", index=False)
    pd.DataFrame(Xte).to_csv(fold_dir / "X_test.csv", index=False)
    pd.DataFrame(ytr).to_csv(fold_dir / "y_train.csv", index=False)
    pd.DataFrame(yte).to_csv(fold_dir / "y_test.csv", index=False)

    print(f"Saved fold {i}")

keywords = fludata["flu_keywords"]
keywords_list = [str(k[0]) for k in keywords.ravel()]
pd.DataFrame(keywords_list, columns=["keyword"]) \
  .to_csv(flu_out / "keywords.csv", index=False)

locs = fludata["flu_locs"]
locs_list = [str(l[0]) for l in locs.ravel()]
pd.DataFrame(locs_list, columns=["location"]) \
  .to_csv(flu_out / "locs.csv", index=False)

#### Amelia → Combine outputs
Combination keeps the original test/train split from influenza_outbreak_dataset.csv:
1. ```train_all_locs.csv```
2. ```test_all_locs.csv```

Additionally, every single feature within X-matrices are also kept.

---
This folder can be zipped and downloaded by running the following:
```
from google.colab import files
!zip -r ak_flucsv_combined.zip /content/SML_PG60/data/processed/"ak_flucsv_combined"
files.download("ak_flucsv_combined.zip")
```



In [4]:

combined_out = repo_dir / "data" / "processed" / "ak_flucsv_combined"
combined_out.mkdir(parents=True, exist_ok=True)

keywords = pd.read_csv(flu_out / "keywords.csv")["keyword"].tolist()
locs = pd.read_csv(flu_out / "locs.csv")["location"].tolist()

train_parts = []
test_parts = []

for i, loc in enumerate(locs):
    fold_dir = flu_out / f"fold_{i:02d}"

    X_train = pd.read_csv(fold_dir / "X_train.csv")
    y_train = pd.read_csv(fold_dir / "y_train.csv")
    X_test = pd.read_csv(fold_dir / "X_test.csv")
    y_test = pd.read_csv(fold_dir / "y_test.csv")

    n_features = X_train.shape[1]
    n_keywords = len(keywords)
    extra_cols = [f"extra_feature{j}" for j in range(n_features - n_keywords)] #keep all features
    feature_cols = keywords + extra_cols

    X_train.columns = feature_cols
    X_test.columns = feature_cols

    train_df = X_train.copy()
    train_df["label"] = y_train.iloc[:, 0].values
    train_df.insert(0, "split", "train")
    train_df.insert(0, "fold_location", loc)
    train_df.insert(0, "fold", i)

    test_df = X_test.copy()
    test_df["label"] = y_test.iloc[:, 0].values
    test_df.insert(0, "split", "test")
    test_df.insert(0, "fold_location", loc)
    test_df.insert(0, "fold", i)

    train_parts.append(train_df)
    test_parts.append(test_df)

train_all = pd.concat(train_parts, ignore_index=True)
test_all = pd.concat(test_parts, ignore_index=True)

train_all.to_csv(combined_out / "train_all_locs.csv", index=False)
test_all.to_csv(combined_out / "test_all_locs.csv", index=False)

print("saved combined csvs.")
print("train shape:", train_all.shape)
print("test shape:", test_all.shape)

saved combined csvs.
train shape: (52560, 549)
test shape: (23280, 549)


### Songhao's conversion
- combines all information from influenza_outbreak_dataset.mat into  ```influenza_outbreak_long.csv``` (>150MB)
- original X matrices within .mat file contain 545 features, however, only 525 named keyword features exist
  * thus ```influenza_outbreak_long.csv``` drops last 20 features within original X matrices
- adds column "time_index" which counts rows in each train/test split per location

In [ ]:
# (Songhao) convert loaded .mat variables to CSV
sg_dir = repo_dir / "data" / "processed" / "flu_csv" / "sg_flucsv"
sg_dir.mkdir(parents=True, exist_ok=True)

csv_output_path = sg_dir / "influenza_outbreak_long.csv"
keywords_output_path = sg_dir / "flu_keywords.csv"
locations_output_path = sg_dir / "flu_locations.csv"

def extract_matlab_string_array(arr):
    values = []
    for item in arr.flatten():
        if isinstance(item, np.ndarray):
            if item.size == 1:
                values.append(str(item.item()).strip())
            else:
                values.append("".join(item.astype(str).flatten()).strip())
        else:
            values.append(str(item).strip())
    return values


def make_safe_unique_names(names):
    safe_names = []
    used = {}

    for name in names:
        clean = (
            str(name)
            .strip()
            .replace(" ", "_")
            .replace("-", "_")
            .replace("/", "_")
            .replace("(", "")
            .replace(")", "")
        )

        if clean == "":
            clean = "unnamed_feature"

        if clean in used:
            used[clean] += 1
            clean = f"{clean}_{used[clean]}"
        else:
            used[clean] = 0

        safe_names.append(clean)

    return safe_names

#-------------------------------#
locs = fludata["flu_locs"]
keywrds = fludata["flu_keywords"]
X_tr = fludata["flu_X_tr"]
y_tr = fludata["flu_Y_tr"]
X_te = fludata["flu_X_te"]
y_te = fludata["flu_Y_te"]
#-------------------------------#

locations = extract_matlab_string_array(locs)
keywords = extract_matlab_string_array(keywrds)
feature_names = make_safe_unique_names(keywords)

all_parts = []

for i, location in enumerate(locations):
    for split_name, X_cell, y_cell in [
        ("train", X_tr, y_tr),
        ("test", X_te, y_te),
    ]:
        X = X_cell[0, i]

        if sparse.issparse(X):
            X = X.toarray()
        else:
            X = np.asarray(X)

        # Keep only the 525 named keyword features --> original X matrices contain 545 features (final 20 columns are dropped)
        X = X[:, :len(feature_names)]

        y = np.asarray(y_cell[0, i]).reshape(-1).astype(int)

        df_part = pd.DataFrame(X, columns=feature_names)
        df_part.insert(0, "location", location)
        df_part.insert(1, "split", split_name)
        df_part.insert(2, "time_index", np.arange(len(y)))
        df_part["label"] = y

        all_parts.append(df_part)

df = pd.concat(all_parts, ignore_index=True)

df.to_csv(csv_output_path, index=False)
pd.DataFrame({"keyword": keywords}).to_csv(keywords_output_path, index=False)
pd.DataFrame({"location": locations}).to_csv(locations_output_path, index=False)

print("Saved main CSV to:", csv_output_path)
print("Saved keywords CSV to:", keywords_output_path)
print("Saved locations CSV to:", locations_output_path)

print("\nData shape:", df.shape)
print("\nSplit counts:")
print(df["split"].value_counts())

print("\nLabel distribution:")
print(df["label"].value_counts())

df.head()

In [ ]:
#Lachlan -> parameter metrics
import torch
import torch.nn as nn

def accuracy_score(preds, y):
  correct = (preds == y).sum()
  return correct / len(y)

def precision_score(preds, y): #preds and y are arrays of 0 and 1
  tp = (preds * y).sum() #Only indicies where preds and y are 1 will be counted
  pred_positives = (preds == 1).sum()
  return tp / (pred_positives + 1e-7) #To prevent div 0 problems

def recall_score(preds, y): #preds and y are arrays of 0 and 1
  tp = (preds * y).sum() #Only indicies where preds and y are 1 will be counted
  real_positives = (y == 1).sum()
  return tp / (real_positives + 1e-7) #To prevent div 0 problems

def f1_score(preds, y):
    prec = precision_score(preds, y)
    rec = recall_score(preds, y)
    return 2 * (prec * rec) / (prec + rec + 1e-7)



In [ ]:
#Lachlan -> basic logistic regression model
class LogisticRegressionModel(nn.Module): #From tute
    def __init__(self, input_dim):
        super(LogisticRegressionModel, self).__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return self.linear(x)

def train_model(model_class, input_dim, criterion_fn, lr, momentum, X_train, y_train, X_eval, y_eval, epochs=50, batch_size=32):
    model = model_class(input_dim)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    #Convert training and evaluation arrays to tensors
    Xt = torch.tensor(X_train, dtype = torch.float32)
    yt = torch.tensor(y_train, dtype = torch.float32).unsqueeze(1)
    Xe = torch.tensor(X_eval, dtype = torch.float32)

    #Set up training data loader
    dataset = torch.utils.data.TensorDataset(Xt, yt)
    train_loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle= True)

    for epoch in range(epochs):
      model.train()
      for batch_X, batch_y in train_loader:
          optimizer.zero_grad()
          predictions = model(batch_X)
          loss = criterion_fn(predictions, batch_y)
          loss.backward()
          optimizer.step()

    model.eval()
    with torch.no_grad():
      outputs = model(Xe) #Determine the raw probabilities
      preds = (torch.sigmoid(outputs) >= 0.5).float().numpy().flatten() #Convert to array of 0 and 1

    acc = accuracy_score(preds, y_eval)
    prec = precision_score(preds, y_eval)
    rec = recall_score(preds, y_eval)
    f1 = f1_score(preds, y_eval)

    return acc, prec, rec, f1


In [ ]:
#Lachlan -> Cross-validated logistic regression
from sklearn.model_selection import StratifiedKFold


#Inputs are np arrays X and y
def cross_val(k_folds, X, y, model, criterion, lr = 0.001, momentum = 1e-2):
  n_features = X.shape[1]
  skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)
  fold_accuracies = []
  fold_precisions = []
  fold_recalls = []
  fold_f1 = []

  folds = skf.split(X, y) #Create training, validation and testing folds

  for fold, (train_idx, val_idx) in enumerate(folds):
      X_train, X_eval = X[train_idx], X[val_idx]
      y_train, y_eval = y[train_idx], y[val_idx]

      acc, prec, rec, f1 = train_model(model, n_features, criterion, lr, momentum, X_train, y_train, X_eval, y_eval, epochs=50)

      fold_accuracies.append(acc)
      fold_precisions.append(prec)
      fold_recalls.append(rec)
      fold_f1.append(f1)

  avg_acc = np.mean(fold_accuracies)
  avg_prec = np.mean(fold_precisions)
  avg_rec = np.mean(fold_recalls)
  avg_f1 = np.mean(fold_f1)

  return avg_acc, avg_prec, avg_rec, avg_f1


In [ ]:
#Lachlan -> run baseline logistic regression

from sklearn.preprocessing import StandardScaler
df = pd.read_csv("influenza_outbreak_long.csv")
df.drop(columns = ['location','split', 'time_index'], inplace = True) #Basic version
df['label'] = df['label'].astype(int) #Cast entries as ints

X_np = (df.drop(columns = ['label'])).values
y_np = df['label'].values


criterion = torch.nn.BCEWithLogitsLoss()

#Configure for cross-validation
k_folds = 5

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_np)
avg_acc,  avg_prec, avg_rec, avg_f1 = cross_val(k_folds, X_scaled, y_np, LogisticRegressionModel, criterion, 0.001, 0.01)

print(f"Averages: Accuracy {avg_acc} | Precision {avg_prec} | Recall {avg_rec} | F1 {avg_f1}")

In [ ]:
#Lachlan -> autoencoder

class FeatureAutoencoder(nn.Module):
    def __init__(self, input_dim, bottleneck_dim):
        super().__init__()

        hidden_dim = max(256, bottleneck_dim * 2)
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, bottleneck_dim),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

def encode_data(X, new_dim, epochs = 50, batch_size = 32, lr = 0.001):
  #Convert to tensor
  Xt = torch.tensor(X, dtype = torch.float32)
  X_dim = Xt.shape[1]

  dataset = torch.utils.data.TensorDataset(Xt)
  data_loader = torch.utils.data.DataLoader(dataset, batch_size = batch_size, shuffle = True)

  autoencoder_model = FeatureAutoencoder(input_dim = X_dim, bottleneck_dim = new_dim)
  criterion = nn.MSELoss()
  #Use Adam for best performance
  optimizer = torch.optim.Adam(autoencoder_model.parameters(), lr = lr)

  autoencoder_model.train()
  for i in range(epochs):
    for (batch_x,) in data_loader:
      reconstructed_data = autoencoder_model(batch_x)
      loss = criterion(reconstructed_data, batch_x)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

  autoencoder = autoencoder_model.encoder
  autoencoder.eval()
  with torch.no_grad():
    reduced_X = autoencoder(Xt)

  new_X_np = reduced_X.detach().cpu().numpy()

  return new_X_np

In [ ]:
#Lachlan -> Determine optimal autoencoding

#X and y must be numpy arrays
#X must be 2D

step_size = 50
def opt_autoencode(k_folds, X, y, epochs = 50, batch_size = 32, lr = 0.001, momentum = 1e-2, step_size):
  X_len = X.shape[1] #Starting length
  test_len = np.floor_divide(X_len, 2)
  best_f1 = 0
  best_len = 0
  best_X = np.empty(X.shape)

  while test_len > 10:
    X_encoded = encode_data(X, test_len, epochs = epochs, batch_size= batch_size, lr = lr)
    criterion = torch.nn.BCEWithLogitsLoss()
    test_f1 = (cross_val(k_folds, X_encoded, y, LogisticRegressionModel,criterion, lr, momentum))[3]
    if test_f1 > best_f1:
      best_len = test_len
      best_X = X_encoded.copy()
    test_len = test_len - 50

  return best_X


In [ ]:
#Lachlan -> Autoencode X
from sklearn.preprocessing import StandardScaler
df = pd.read_csv("influenza_outbreak_long.csv")
df.drop(columns = ['location','split', 'time_index'], inplace = True) #Basic version
df['label'] = df['label'].astype(int) #Cast entries as ints

scaler = StandardScaler()
X_np = scaler.fit_transform((df.drop(columns = ['label'])).values)
y_np = df['label'].values
k_folds = 5
epochs = 30
batch_size = 32
lr = 0.001
momentum = 1e-2

X_reduced = opt_autoencode(k_folds, X_np, y_np, epochs, batch_size, lr, momentum)
print(X_reduced.shape)

In [ ]:
#Lachlan -> Copied Amelia's state filtering
c_df = pd.read_csv(repo_dir / "data/covid19_tweets.csv.xz")
import re

c_df["user_location"] = c_df["user_location"].fillna("").astype(str).str.strip().str.lower()
# dictionaries
abbr_to_state = {
    "al": "alabama","ak": "alaska", "az": "arizona", "ar": "arkansas", "ca": "california","co": "colorado",
    "ct": "connecticut", "de": "delaware", "fl": "florida", "ga": "georgia", "hi": "hawaii", "id": "idaho",
    "il": "illinois", "in": "indiana", "ia": "iowa", "ks": "kansas", "ky": "kentucky", "la": "louisiana",
    "me": "maine", "md": "maryland", "ma": "massachusetts", "mi": "michigan", "mn": "minnesota",
    "ms": "mississippi", "mo": "missouri", "mt": "montana", "ne": "nebraska", "nv": "nevada",
    "nh": "new hampshire", "nj": "new jersey", "nm": "new mexico", "ny": "new york",
    "nc": "north carolina", "nd": "north dakota", "oh": "ohio", "ok": "oklahoma", "or": "oregon",
    "pa": "pennsylvania", "ri": "rhode island", "sc": "south carolina", "sd": "south dakota",
    "tn": "tennessee", "tx": "texas", "ut": "utah", "vt": "vermont", "va": "virginia",
    "wa": "washington", "wv": "west virginia", "wi": "wisconsin", "wy": "wyoming", "dc": "district of columbia"
}
state_names = set(abbr_to_state.values())

usa_terms = {"usa", "us", "united states", "america", "u.s.", "u.s.a."}

city_to_state = {
    "new york": "new york", "nyc": "new york", "brooklyn": "new york", "manhattan": "new york",
    "los angeles": "california", "san diego": "california", "san francisco": "california", "sacramento": "california", "long beach": "california",
    "houston": "texas", "dallas": "texas", "austin": "texas",
    "miami": "florida", "orlando": "florida",
    "chicago": "illinois",
    "atlanta": "georgia",
    "boston": "massachusetts",
    "las vegas": "nevada",
    "seattle": "washington",
    "new orleans": "louisiana",
    "st louis": "missouri", "saint louis": "missouri",
    "washington dc": "district of columbia"
}

# specific non-us locations/terms to exclude
junk = ["everywhere", "worldwide", "23 countries", "opt-out", "catch me", "the beach",
        "in the vineyard", "available now", "working"]

foreign = ["canada", "uk", "australia", "india", "germany", "france", "italy", "spain", "brazil",
           "argentina", "china", "japan", "south korea", "paris", "london", "vienna", "delhi", "rio",
           "beijing", "nairobi", "joburg"]

bad_ab = {"in", "or", "me", "hi"}

def is_ambig(loc): # multiple locations or terms foreign to usa
    separators = [",", ";", "|", "/", " and ", " & "]
    foreign_count = 0
    for term in foreign:
        if term in loc:
            foreign_count += 1
    if foreign_count >= 1 and any (sep in loc for sep in separators):
        return True
    return False

def is_junk(loc):
    for term in junk:
        if term in loc:
            return True
    return False

#filter
def get_state(loc):
    loc = loc.lower().strip()

    if loc == "":
        return None

    if is_junk(loc):
        return None

    if is_ambig(loc):
        return None

    for state in state_names:
        if state in loc:
            return state

    tokens = re.findall(r"\b[a-z]{2}\b", loc)
    for token in tokens:
        if token in abbr_to_state and token not in bad_ab:
            return abbr_to_state[token]

    for city in city_to_state:
        if city in loc:
            return city_to_state[city]
    return None

# confidence levels (for later analysis.. noise filtering, weighting, etc.)
def loc_conf(loc):
    loc = loc.lower().strip()
    for state in state_names:
        if state in loc:
            return "high"
    tokens = re.findall(r"\b[a-z]{2}\b", loc)
    for token in tokens:
        if token in abbr_to_state and token not in bad_ab:
            return "high"
    for city in city_to_state:
        if city in loc:
            return "medium"
    for term in usa_terms:
        if term in loc:
            return "low"
    return "none"

c_df["state"] = c_df["user_location"].apply(get_state)
c_df["state_confidence"] = c_df["user_location"].apply(loc_conf)
state_cdf = c_df[c_df["state"].notna()].copy()
print(state_cdf[["user_location", "state", "state_confidence"]].head(100))
print(len(state_cdf), "rows with identified US state locations")

In [ ]:
#Lachlan -> Convert tweet text to dictionary using first dataset keywords
flu_df = pd.read_csv("influenza_outbreak_long.csv")
flu_columns = flu_df.columns.tolist()
columns_to_drop = ['location', 'split', 'time_index', 'label']
flu_keywords = [item for item in flu_columns if item not in columns_to_drop]

def vectorise_tweet(tweet):
  text = tweet.lower()
  counts = {}
  for keyword in flu_keywords:
    count = text.count(keyword)
    counts[keyword] = count
  return counts



In [ ]:
#Lachlan -> Sort COVID tweets by whether or not the 'covid19' hashtag appears.

covid_df = pd.read_csv('covid19_tweets.csv')
word_count_series = covid_df['text'].apply(vectorise_tweet)
covid_count_df = pd.DataFrame(list(word_count_series))
covid_vector_df = covid_count_df.reindex(columns = flu_df.columns, fill_value = 0)
covid_vector_df.head(20)
